In [ ]:
import os
import pandas as pd

if not os.path.exists('data/ml-latest-small'):
    os.makedirs('data', exist_ok=True)
    # Using shell commands (!) to download and unzip the dataset quietly (-q)
    !curl -O https://files.grouplens.org/datasets/movielens/ml-1m.zip
    !unzip -q ml-1m.zip -d data/
    !rm ml-1m.zip
    print("Dataset downloaded and extracted successfully!")

In [ ]:
!pwd
!ls
!ls data
!ls data/ml-1m
!cat data/ml-1m/movies.dat | head -n 5

In [ ]:
# The 1M dataset uses '::' as a separator and has no header row
users_cols = ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
users_df = pd.read_csv('./data/ml-1m/users.dat', sep='::', engine='python', names=users_cols, encoding='latin-1')

print("User Demographics Sample:")
display(users_df.head())

movies_cols = ['MovieID', 'Title', 'Genres']
movies_df = pd.read_csv('./data/ml-1m/movies.dat', sep='::', engine='python', names=movies_cols, encoding='latin-1')

print("\nMovie Features Sample:")
display(movies_df.head())

ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
ratings_df = pd.read_csv('./data/ml-1m/ratings.dat', sep='::', engine='python', names=ratings_cols, encoding='latin-1')

print(f"\nTotal Ratings: {len(ratings_df)}")

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import numpy as np

merged_df = ratings_df.merge(users_df, on='UserID').merge(movies_df, on='MovieID')

merged_df['Gender_Idx'] = LabelEncoder().fit_transform(merged_df['Gender'])
merged_df['Age_Idx'] = LabelEncoder().fit_transform(merged_df['Age'])

merged_df['Occupation_Idx'] = merged_df['Occupation']

merged_df['Genres_List'] = merged_df['Genres'].str.split('|')
mlb = MultiLabelBinarizer()
genres_multi_hot = mlb.fit_transform(merged_df['Genres_List'])
merged_df['Genres_Encoded'] = list(genres_multi_hot)

In [ ]:
merged_df.head()

In [ ]:

class MovieLensDataset(Dataset):
    def __init__(self, df):
        self.gender = torch.tensor(df['Gender_Idx'].values, dtype=torch.long)
        self.age = torch.tensor(df['Age_Idx'].values, dtype=torch.long)
        self.occupation = torch.tensor(df['Occupation_Idx'].values, dtype=torch.long)
        self.genres = torch.tensor(np.stack(df['Genres_Encoded'].values), dtype=torch.float32)

        self.ratings = torch.tensor(df['Rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        user_features = {
            'gender': self.gender[idx],
            'age': self.age[idx],
            'occupation': self.occupation[idx],
        }
        return user_features, self.genres[idx], self.ratings[idx]


full_dataset = MovieLensDataset(merged_df)
sample_user, sample_movie, sample_rating = full_dataset[0]

print("Target Rating:", sample_rating.item())
print("User Features:", sample_user)
print("Movie Features (Genres Multi-Hot Shape):", sample_movie.shape)

In [ ]:
import torch.nn as nn

class UserTower(nn.Module):
    def __init__(self, embedding_dim=16, output_dim=32):
        super().__init__()
        self.age_emb = nn.Embedding(num_embeddings=7, embedding_dim=embedding_dim)
        self.gender_emb = nn.Embedding(num_embeddings=2, embedding_dim=embedding_dim)
        self.occ_emb = nn.Embedding(num_embeddings=21, embedding_dim=embedding_dim)
        
        self.fc1 = nn.Linear(embedding_dim * 3, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, user_features):
        age_vec = self.age_emb(user_features['age'])
        gender_vec = self.gender_emb(user_features['gender'])
        occ_vec = self.occ_emb(user_features['occupation'])
        
        x = torch.cat([age_vec, gender_vec, occ_vec], dim=1)
        x = self.relu(self.fc1(x))
        return self.fc2(x) 


class MovieTower(nn.Module):
    def __init__(self, num_genres=18, output_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(num_genres, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, movie_features):
        x = self.relu(self.fc1(movie_features))
        return self.fc2(x) 


class TwoTowerRecommender(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_tower = UserTower()
        self.movie_tower = MovieTower()

    def forward(self, user_features, movie_features):
        # 1. Pass data through the towers
        v_u = self.user_tower(user_features)
        v_m = self.movie_tower(movie_features)
        
        prediction = torch.sum(v_u * v_m, dim=1)
        
        return prediction

model = TwoTowerRecommender()
print(model)